In [3693]:
from multiprocessing import Pool
import os, sys, time
import numpy as np
from pymatching import Matching
import random as rnd
from itertools import combinations
from scipy.sparse import coo_matrix
import gurobipy as gp
from gurobipy import GRB

In [3694]:
from tableaux_old import (
    connection_dict,
    star_edges_of_vertex,
    get_vertex_row,
    get_vertex_column,
    apply_Z,
    is_A_LX_commute,
    z_logical_check_matrix,
    measure_A
)
from D4_old import D4_Code


In [3695]:
l = 2
Nx = Ny = 3 * l

cn_dict = connection_dict((Nx, Ny))

encode_x = np.array([0,1,2], dtype=int) #0,1,2

#initialize in +~ state
code = D4_Code(
    l=l,
    encode_x=encode_x,
    cn_dict=cn_dict,
    V=None,
    E1_list=None,
    E2_list=None,
    Gamma1=None,
    Gamma2=None,
    w1_arr=None,
    w2_arr=None,
)

In [3696]:
np.nonzero(cn_dict['HH_log_x'][2])

(array([  1,  14,  19,  32,  43,  61,  85, 103]),)

In [3697]:
print(code.LX_sign) #verticals then horizontals, 0 mean eigenval 1, 2 means NA

[0. 0. 0. 2. 2. 2.]


In [3698]:
vertex = 14
print(get_vertex_column(vertex, Nx))
print(get_vertex_row(vertex, Nx, Ny))

2
2


In [3699]:
star_edges = star_edges_of_vertex(vertex, (Nx, Ny))
star_edges

array([79, 13, 55, 92, 21, 50])

In [3700]:
code.LX_sign

array([0., 0., 0., 2., 2., 2.])

In [3701]:
code.measure_e_anyons()

([[], [], []], [[], [], []])

In [3702]:
is_A_LX_commute(linear_size=(Nx,Ny),
                kv=21,
                LX_vec=code.LX_vec,
                LX_sign=code.LX_sign,
                LZ_state=code.LZ,
                bL_state=code.bL,
                bR_state=code.bR,
                cn_dict=code.cn_dict)

array([0., 0., 0., 2., 2., 2.])

In [3703]:
measure_A(linear_size=(Nx,Ny),
                kv=21,
                SS=code.SS,
                DD=code.DD,
                RR=code.RR,
                LX_vec=code.LX_vec,
                LX_sign=code.LX_sign,
                LZ_state=code.LZ,
                bL_state=code.bL,
                bR_state=code.bR,
                cn_dict=code.cn_dict)

np.float64(0.0)

In [3704]:

def apply_logical_Z(code, z_ind: int):
    """
    Apply logical Z specified by row z_ind of cn_dict['HH_log_z'].
    z_ind uses ordering [Rv,Gv,Bv,Rh,Gh,Bh,(...)'].
    """
    support = np.nonzero(code.cn_dict["HH_log_z"][z_ind, :])[0]  # :contentReference[oaicite:5]{index=5}
    for e in support:
        apply_Z(
            int(e),
            (code.Nx, code.Ny),
            code.SS, code.RR,
            code.LX_vec, code.LX_sign,
            code.cn_dict,
        )  # same signature as used in Z_errors :contentReference[oaicite:6]{index=6}

import numpy as np

def apply_undecorated_logical_X(code, x_ind: int):
    """
    Parameters
    ----------
    code : D4_Code
        Initialized code object.
    x_ind : int
        Row index into code.cn_dict["HH_log_x"] selecting which logical-X to apply.
        (This is the same indexing used for the code's bare logical-X supports.)
    verify : bool
        If True, assert that bL and bR return to all-zeros at the end.

    Returns
    -------
    support_edges : np.ndarray
        The edge indices (k) on which X was applied.
    """
    HH_log_x = code.cn_dict["HH_log_x"]
    support_edges = np.flatnonzero(HH_log_x[x_ind, :]).astype(int)

    # Apply X on each edge in the logical string support
    for k in support_edges:
        code.single_edge_X(k)  # wraps apply_X and updates tableau/logicals


    return support_edges



In [3705]:
apply_undecorated_logical_X(code, 4) #green horizontal
# apply_undecorated_logical_X(code, 4)
# apply_logical_Z(code, 3)
# apply_Z(21,(Nx,Ny),code.SS,code.RR,code.LX_vec,code.LX_sign,code.cn_dict)
# code.single_edge_X(86)

array([  0,   3,  37,  40,  74,  77, 102, 105])

In [3706]:
# apply_logical_Z(code, 3)

In [3707]:
code.LZ

array([0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.])

In [3713]:
code.measure_e_anyons()

([[], [], []], [[], [], [5, 30, 5, 30, 5, 30]])

In [3714]:
code.LX_sign

array([2., 0., 2., 2., 2., 2.])

In [3710]:
# code.cn_dict

In [3711]:
is_A_LX_commute(linear_size=(Nx,Ny),
                kv=4,
                LX_vec=code.LX_vec,
                LX_sign=code.LX_sign,
                LZ_state=code.LZ,
                bL_state=code.bL,
                bR_state=code.bR,
                cn_dict=code.cn_dict)

array([2., 0., 0., 2., 2., 2.])